In [ ]:
import numpy as np
import torch
import torch.nn as nn
import h5py
import matplotlib.pyplot as plt

import os
import glob

from dotenv import load_dotenv

from models.resnet_rnn import ResnetRNN
from models.resnet_rnn_stated import ResnetRNNStated
from models.residual_block import ResidualBlock


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

print("This can be run after the creation of Test/Train and Processed-Recordings.")

In [2]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))
MODEL_DIMS = USE_DVS + USE_APS
if MODEL_DIMS == 0: raise Exception("MODEL_DIMS must be non-zero")

BINSIZE_MS = int(os.getenv("BINSIZE_MS"))
HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"

BATCH_SIZE = int(os.getenv("BATCH_SIZE"))

# --- Location parameters ---
MODEL_SAVE_DIR = os.getenv("MODEL_SAVE_DIR")
DATASET_DIR = os.getenv("OUT_DIR")
TRAINING_DIR = f"{DATASET_DIR}/Training"
TEST_DIR = f"{DATASET_DIR}/Test"
PRUNED_DATASET_DIR = TRAINING_DIR + "/Pruned-Recordings"

IMAGES_DIR = "/srv/scratch/A2RoboRes/OltansDatasets/img/original_preprocess"

In [3]:
criterion = nn.MSELoss()
def generate_predictions(model, file_path, batch_size=BATCH_SIZE):
    inv_apsdvs_max = 1 / 255
    def normalise(data):
            # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
            new_data = np.empty_like(data, dtype=DTYPE)
            new_data = (data * inv_apsdvs_max).astype(DTYPE)
            return new_data

    model.eval()

    with h5py.File(file_path, "r") as f:
        dvs_frame = np.array(f[DVS_FRAME_NAME])
        data_len = dvs_frame.shape[0]
        aps_frame = np.array(f[APS_FRAME_NAME])
        steering_angles = np.array(f["steering_wheel_angle"])
        if USE_APS and USE_DVS:
            # we have batch x 224 x 224 x 2 after stack
            combined_frames = np.stack((dvs_frame, aps_frame), axis=3)
        else:
            combined_frames = aps_frame if USE_APS else dvs_frame
            combined_frames = np.expand_dims(combined_frames, axis=3)
        

        preds, gts = [], []
        for start in range(0, data_len, batch_size):
            end = min(start + batch_size, data_len)
            batch_data = combined_frames[start:end]
            batch_data = normalise(batch_data)
            # b, h, w, c = batch_data.shape
            # transformed_batch = np.empty((b, c, h, w), dtype=DTYPE)

            # shuffle dimensions 
            batch_data = np.transpose(batch_data, (0, 3, 1, 2))
            batch_data = torch.from_numpy(batch_data).to(device)
            
            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    batch_out = model(batch_data).cpu().numpy()
                preds.append(batch_out)
                gts.append(steering_angles[start:end])

        preds = np.concatenate(preds, axis=0)
        gts = np.concatenate(gts, axis=0)
        # Convert to torch tensors
        preds_tensor = torch.tensor(preds).to(device)
        gts_tensor = torch.tensor(gts).to(device)

        # Calculate loss
        loss = criterion(preds_tensor, gts_tensor)

        return preds, gts, loss

In [4]:
model_files = glob.glob(MODEL_SAVE_DIR + "/*.pth")
best_model_dir = f"{MODEL_SAVE_DIR}/best/*.pth"

# Don't expand with last, since it can change if training is going on in parallel.
# last_model_dir = f"{MODEL_SAVE_DIR}/last/*.pth"
# model_files.extend(glob.glob(last_model_dir))

model_files = sorted(model_files, key=os.path.getctime)
model_files.extend(glob.glob(best_model_dir))
        
train_files = sorted(glob.glob(PRUNED_DATASET_DIR + "/*.hdf5"))
test_files = sorted(glob.glob(TEST_DIR + "/*.hdf5"))

In [5]:
# Delete old images
try:
        os.mkdir(IMAGES_DIR)
except FileExistsError:
    for img in glob.glob(IMAGES_DIR + "/*"):
        os.remove(img)

In [6]:
def plot_predictions(model, files, model_files, model_name, dataset_name):
    fig = plt.figure(figsize=(15, 7.5 * len(files)))
    plt.suptitle(f"Predictions for the {dataset_name} Set by Model ({model_name})")
    for i, f in enumerate(files):
        announce_str = f"Working on {f}"
        print(f"{announce_str}\n{'-' * len(announce_str)}")

        ax = plt.subplot(len(files), 1, i + 1)
        ax.set_title(f, fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel("Predicted Values", fontsize=12)
        ax.grid(True)

        for model_file in model_files:
            epoch = model_file.split('_')[0]
            epoch = epoch.split('/')[-1][1:]
            model_dict = torch.load(model_file, weights_only=True, map_location=device)
            model.load_state_dict(model_dict["model_state_dict"])
            model.to(device)
            
            preds, gt, loss = generate_predictions(model, f)
            print(f"RMSE for model {epoch} is {loss ** 0.5:.2f}")
            time_axis = [x * (BINSIZE_MS / 1000) for x in range(len(preds))]
            ax.plot(time_axis, preds, label=f"E:{epoch}, L:{loss ** 0.5:.2f}")
        
        ax.plot(time_axis, gt, label="truth", linewidth=2, linestyle='--')
        l = ax.legend(loc='best')
        # add the "best" legend, second last after ground truth, as it is added to the end with extend
        current_text = l.get_texts()[-2].get_text()
        l.get_texts()[-2].set_text(f"Best: {current_text}")

    plt.tight_layout()
    svg_path = os.path.join(IMAGES_DIR, f"{model_name}_{dataset_name}.svg")
    fig.savefig(svg_path, format="svg")
    print(f"Saved plot to {svg_path}")
    plt.show()
    plt.close(fig)

In [ ]:
# Unstated Model
model = ResnetRNN(ResidualBlock, [3, 4, 6, 3], num_channels=MODEL_DIMS)
plot_predictions(model, test_files, model_files, "ResnetRNN", "Test")       

In [ ]:
plot_predictions(model, train_files, model_files, "ResnetRNN", "Training")

In [ ]:
# Stated Model
model = ResnetRNNStated(ResidualBlock, [3, 4, 6, 3], num_channels=MODEL_DIMS)
plot_predictions(model, test_files, model_files, "ResnetRNNStated", "Test")

In [ ]:
plot_predictions(model, train_files, model_files, "ResnetRNNStated", "Training")